In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm  # Training progress bar

# Project root link karna
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from models.segmentation.dataset import DeepGlobeDataset
from torchvision.models.segmentation import deeplabv3_resnet50

# 1. Force Device selection to GPU (RTX 3050 Laptop GPU!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Training will run on: {device}")

# 2. Initialize Model and move it to the GPU
model = deeplabv3_resnet50(num_classes=7)
model = model.to(device)
print(f"🤖 DeepLabV3-ResNet50 successfully loaded on: {device}")

# 3. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 4. Example DataLoader setup (Keep batch size small for 4GB VRAM)
# Replace with your actual dataset paths/arguments as needed:
# train_dataset = DeepGlobeDataset(...)
# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)

print("✨ All systems ready! You can now write your training loop.")

🚀 Training will run on: cuda
🤖 DeepLabV3-ResNet50 successfully loaded on: cuda
✨ All systems ready! You can now write your training loop.


In [2]:
import os
from torch.utils.data import DataLoader
from models.segmentation.dataset import DeepGlobeDataset

# 1. Point directly to your train folder
TRAIN_DIR = r"D:\project data\AI_afforestation\datasets\DeepGlobeDataset\train"

# 2. Instantiate your dataset (passing TRAIN_DIR for both since images and masks are in the same folder)
train_dataset = DeepGlobeDataset(
    image_dir=TRAIN_DIR,
    mask_dir=TRAIN_DIR
)

# 3. Create the train_loader (Batch size 2 is safe for your 4GB VRAM)
train_loader = DataLoader(
    train_dataset, 
    batch_size=2,        
    shuffle=True, 
    num_workers=0        # Keep 0 on Windows
)

print(f"📦 Train DataLoader created successfully with {len(train_dataset)} samples!")

📦 Train DataLoader created successfully with 803 samples!


In [ ]:
model.train()
for images, masks in train_loader:
    # Move data to GPU
    images = images.to(device)
    masks = masks.to(device)
    
    optimizer.zero_grad()
    outputs = model(images)['out']
    loss = criterion(outputs, masks)
    
    loss.backward()
    optimizer.step()

In [ ]:
from torch.utils.data import DataLoader

# Create the train_loader with drop_last=True to avoid batch size of 1
train_loader = DataLoader(
    train_dataset, 
    batch_size=2,        
    shuffle=True, 
    num_workers=0,
    drop_last=True       # <-- This fixes the batch size = 1 crash!
)

print(f"📦 Train DataLoader updated successfully!")

📦 Train DataLoader updated successfully!


In [2]:
import os

project_root = "."

for root, dirs, files in os.walk(project_root):
    level = root.replace(project_root, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = ' ' * 4 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

./
    01_dataset_exploration.ipynb
    02_model_training.ipynb
    03_inference.ipynb
    04_tree_recommendation.ipynb
    05_survival_prediction.ipynb
    06_carbon_credits.ipynb
    _03.1_inference.ipynb
    checkpoints/
        ml_encoders.pkl
        survival_regressor.pkl
        tree_recommender.pkl
    datasets/
        master/
            master_dataset.csv


In [3]:
import os

print("=" * 50)
print("PROJECT SUMMARY")
print("=" * 50)

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith((".py", ".ipynb", ".yaml", ".yml", ".json")):
            print(os.path.join(root, file))

PROJECT SUMMARY
.\01_dataset_exploration.ipynb
.\02_model_training.ipynb
.\03_inference.ipynb
.\04_tree_recommendation.ipynb
.\05_survival_prediction.ipynb
.\06_carbon_credits.ipynb
.\_03.1_inference.ipynb


In [5]:
import torch

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2), "GB")

Torch Version: 2.5.1+cu121
CUDA Available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.0 GB


In [6]:
import os

with open("project_report.txt", "w", encoding="utf-8") as f:

    f.write("PROJECT STRUCTURE\n")
    f.write("="*50 + "\n")

    for root, dirs, files in os.walk("."):
        f.write(f"\n{root}\n")
        for file in files:
            f.write(f"  {file}\n")

print("Report generated: project_report.txt")

Report generated: project_report.txt


In [8]:
import os
import pandas as pd

ROOT_DIR = os.path.abspath("..")

print("Current Path:", os.getcwd())
print("Root Path:", ROOT_DIR)

for root, dirs, files in os.walk(ROOT_DIR):
    if "master_dataset.csv" in files:
        print("\nDataset Found:")
        print(os.path.join(root, "master_dataset.csv"))

Current Path: d:\project data\AI_afforestation\notebooks
Root Path: d:\project data\AI_afforestation

Dataset Found:
d:\project data\AI_afforestation\datasets\master\master_dataset.csv

Dataset Found:
d:\project data\AI_afforestation\notebooks\datasets\master\master_dataset.csv


In [9]:
import pandas as pd

df = pd.read_csv("../datasets/master/master_dataset.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.info())

(1200, 14)
['Image_ID', 'Land_Type', 'Vegetation_Percentage', 'Water_Nearby', 'Soil_Type', 'pH', 'Moisture', 'Temperature', 'Rainfall', 'Humidity', 'AQI', 'Elevation', 'Best_Tree', 'Survival_Probability']
  Image_ID    Land_Type  Vegetation_Percentage  Water_Nearby Soil_Type  \
0   img001  Agriculture              38.762885             0     Loamy   
1   img002        Urban              33.309710             0     Rocky   
2   img003       Forest              81.690174             1    Clayey   
3   img004        Urban              19.749674             0     Loamy   
4   img005        Urban              12.226957             0     Loamy   

         pH   Moisture  Temperature     Rainfall   Humidity  AQI    Elevation  \
0  6.742423  76.008795    24.129228   803.013863  29.435005  197  1302.151836   
1  8.373838  52.726324    17.593725   257.132719  54.725283   54   110.033809   
2  7.123341  68.448165    15.042850  1093.774188  31.837273  160  1461.415414   
3  7.315269  59.724605    

In [10]:
import joblib

tree_model = joblib.load("../checkpoints/tree_recommender.pkl")
survival_model = joblib.load("../checkpoints/survival_regressor.pkl")

print(type(tree_model))
print(type(survival_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [14]:
import os
import sys
import pandas as pd
import sklearn
import numpy as np

print("="*50)
print("SYSTEM REPORT")
print("="*50)

print("Python :", sys.version)
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Sklearn:", sklearn.__version__)

print("\nCurrent Directory")
print(os.getcwd())

print("\nFiles Present")
for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root, file))

SYSTEM REPORT
Python : 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
Pandas : 3.0.3
NumPy  : 2.4.6
Sklearn: 1.9.0

Current Directory
d:\project data\AI_afforestation\notebooks

Files Present
.\01_dataset_exploration.ipynb
.\02_model_training.ipynb
.\03_inference.ipynb
.\04_tree_recommendation.ipynb
.\05_survival_prediction.ipynb
.\06_carbon_credits.ipynb
.\project_report.txt
.\_03.1_inference.ipynb
.\checkpoints\ml_encoders.pkl
.\checkpoints\survival_regressor.pkl
.\checkpoints\tree_recommender.pkl
.\datasets\master\master_dataset.csv


In [15]:
df = pd.read_csv("../datasets/master/master_dataset.csv")

print(df.shape)
print(df.columns)
print(df.head(3))

(1200, 14)
Index(['Image_ID', 'Land_Type', 'Vegetation_Percentage', 'Water_Nearby',
       'Soil_Type', 'pH', 'Moisture', 'Temperature', 'Rainfall', 'Humidity',
       'AQI', 'Elevation', 'Best_Tree', 'Survival_Probability'],
      dtype='str')
  Image_ID    Land_Type  Vegetation_Percentage  Water_Nearby Soil_Type  \
0   img001  Agriculture              38.762885             0     Loamy   
1   img002        Urban              33.309710             0     Rocky   
2   img003       Forest              81.690174             1    Clayey   

         pH   Moisture  Temperature     Rainfall   Humidity  AQI    Elevation  \
0  6.742423  76.008795    24.129228   803.013863  29.435005  197  1302.151836   
1  8.373838  52.726324    17.593725   257.132719  54.725283   54   110.033809   
2  7.123341  68.448165    15.042850  1093.774188  31.837273  160  1461.415414   

  Best_Tree  Survival_Probability  
0   Amaltas                 94.22  
1   Amaltas                 85.21  
2   Amaltas              

In [16]:
import pandas as pd

df = pd.read_csv("./datasets/master/master_dataset.csv")

print(df.shape)

(1200, 14)


In [17]:
X = df.drop([
    "Image_ID",
    "Best_Tree",
    "Survival_Probability"
], axis=1)

y_tree = df["Best_Tree"]

In [18]:
y_survival = df["Survival_Probability"]

In [19]:
import joblib

encoders = joblib.load("./checkpoints/ml_encoders.pkl")

print(type(encoders))
print(encoders)

<class 'dict'>
{'land': LabelEncoder(), 'soil': LabelEncoder(), 'tree': LabelEncoder(), 'features': ['Vegetation_Percentage', 'Water_Nearby', 'Land_Type_Enc', 'Soil_Type_Enc', 'pH', 'Moisture', 'Temperature', 'Rainfall', 'Humidity', 'AQI', 'Elevation']}


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_tree, y_test_tree = train_test_split(
    X,
    y_tree,
    test_size=0.2,
    random_state=42
)

_, _, y_train_surv, y_test_surv = train_test_split(
    X,
    y_survival,
    test_size=0.2,
    random_state=42
)

In [22]:
!pip install seaborn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import joblib

tree_model = joblib.load("./checkpoints/tree_recommender.pkl")
survival_model = joblib.load("./checkpoints/survival_regressor.pkl")

In [24]:
print("Tree Features:")
print(tree_model.feature_names_in_)

print()

print("Survival Features:")
print(survival_model.feature_names_in_)

Tree Features:
['Vegetation_Percentage' 'Water_Nearby' 'Land_Type_Enc' 'Soil_Type_Enc'
 'pH' 'Moisture' 'Temperature' 'Rainfall' 'Humidity' 'AQI' 'Elevation']

Survival Features:
['Vegetation_Percentage' 'Water_Nearby' 'Land_Type_Enc' 'Soil_Type_Enc'
 'pH' 'Moisture' 'Temperature' 'Rainfall' 'Humidity' 'AQI' 'Elevation']
